# 04 olap queries

Queries analíticas OLAP

In [19]:

import pandas as pd
import sys
import os
from sqlalchemy import create_engine, text


sys.path.append(os.path.abspath('..'))
from config import engine

print("Pandas e Engine carregados com sucesso.")

Pandas e Engine carregados com sucesso.


In [20]:

# Exercício 1: Lucro total por gerente regional no ano de 2014
query_ex1 = """
SELECT 
    g.nome_gerente,
    g.regiao,
    ROUND(SUM(f.vlr_lucro), 2) AS lucro_total
FROM FATO_VENDAS f
JOIN DIM_GERENTE g ON f.sk_gerente = g.sk_gerente
JOIN DIM_TEMPO t ON f.sk_tempo = t.sk_tempo
WHERE t.ano = 2014
GROUP BY g.nome_gerente, g.regiao
ORDER BY lucro_total DESC;
"""
pd.read_sql(query_ex1, engine)

,nome_gerente,regiao,lucro_total
0,Anna Andreadi,West,20065.69
1,Chuck Magee,East,17059.61
2,Cassandra Brandow,South,11879.12
3,Kelly Williams,Central,539.55


In [21]:

# Exercício 2: Taxa de devolução por subcategoria (Home Office)
query_ex2 = """
SELECT 
    p.subcategoria,
    COUNT(DISTINCT CASE WHEN f.flag_retorno = 1 THEN f.nk_pedido END) AS pedidos_devolvidos,
    COUNT(DISTINCT f.nk_pedido) AS total_pedidos,
    ROUND(
        COUNT(DISTINCT CASE WHEN f.flag_retorno = 1 THEN f.nk_pedido END) / 
        COUNT(DISTINCT f.nk_pedido) * 100, 2
    ) AS taxa_devolucao_pct
FROM FATO_VENDAS f
JOIN DIM_PRODUTO p ON f.sk_produto = p.sk_produto
JOIN DIM_CLIENTE c ON f.sk_cliente = c.sk_cliente
WHERE c.segmento = 'Home Office'
GROUP BY p.subcategoria
ORDER BY taxa_devolucao_pct DESC;
"""
pd.read_sql(query_ex2, engine)

,subcategoria,pedidos_devolvidos,total_pedidos,taxa_devolucao_pct
0,Supplies,3,29,10.34
1,Labels,6,66,9.09
2,Machines,2,22,9.09
3,Phones,13,151,8.61
4,Chairs,7,98,7.14
5,Binders,16,237,6.75
6,Furnishings,11,164,6.71
7,Bookcases,2,32,6.25
8,Paper,14,224,6.25
9,Appliances,4,67,5.97


In [22]:

# Exercício 3: Subtotais Ano > Trimestre > Mês
query_ex3 = """
SELECT 
    COALESCE(CAST(t.ano AS CHAR), 'TOTAL GERAL') AS ano,
    COALESCE(CAST(t.trimestre AS CHAR), 'Subtotal Ano') AS trimestre,
    COALESCE(CAST(t.mes AS CHAR), 'Subtotal Trimestre') AS mes,
    COUNT(DISTINCT f.nk_pedido) AS volume_pedidos,
    ROUND(SUM(f.vlr_vendas), 2) AS receita_total
FROM FATO_VENDAS f
JOIN DIM_TEMPO t ON f.sk_tempo = t.sk_tempo
GROUP BY t.ano, t.trimestre, t.mes WITH ROLLUP
ORDER BY 
    t.ano IS NULL, t.ano, 
    t.trimestre IS NULL, t.trimestre, 
    t.mes IS NULL, t.mes;
"""
pd.read_sql(query_ex3, engine)

,ano,trimestre,mes,volume_pedidos,receita_total
0,TOTAL GERAL,Subtotal Ano,Subtotal Trimestre,5009,2297200.86
1,2014,Subtotal Ano,Subtotal Trimestre,969,484247.50
2,2014,1,Subtotal Trimestre,131,74447.80
3,2014,1,1,32,14236.90
4,2014,1,2,28,4519.89
...,...,...,...,...,...
64,2017,3,9,226,87866.65
65,2017,4,Subtotal Trimestre,632,280054.07
66,2017,4,10,147,77776.92
67,2017,4,11,261,118447.83


In [23]:
# Exercício 4: Ticket Médio ponderado por modalidade e segmento
query_ex4 = """
SELECT 
    e.modalidade_envio,
    c.segmento,
    COUNT(DISTINCT f.nk_pedido) AS pedidos,
    ROUND(SUM(f.vlr_vendas), 2) AS receita,
    ROUND(SUM(f.vlr_vendas) / COUNT(DISTINCT f.nk_pedido), 2) AS ticket_medio
FROM FATO_VENDAS f
JOIN DIM_ENVIO e ON f.sk_envio = e.sk_envio
JOIN DIM_CLIENTE c ON f.sk_cliente = c.sk_cliente
GROUP BY e.modalidade_envio, c.segmento
ORDER BY ticket_medio DESC;
"""
display(pd.read_sql(query_ex4, engine))


,modalidade_envio,segmento,pedidos,receita,ticket_medio
0,Same Day,Corporate,58,45121.32,777.95
1,First Class,Home Office,141,86400.99,612.77
2,Second Class,Home Office,165,81568.58,494.36
3,Second Class,Corporate,301,146126.04,485.47
4,Second Class,Consumer,498,231498.95,464.86
5,Standard Class,Consumer,1535,710137.07,462.63
6,Same Day,Home Office,49,22645.44,462.15
7,Standard Class,Corporate,905,409040.54,451.98
8,Standard Class,Home Office,554,239038.14,431.48
9,First Class,Corporate,250,105858.47,423.43


Dá para ver pelos números que o jeito de envio se é rápido ou demorado quase não muda o valor médio das compras. O que realmente faz diferença no tamanho do pedido é o tipo de cliente. Clientes Corporate ou Home Office costumam comprar muito mais de uma vez só, não importa se o frete é rápido ou não. Ou seja, o que define o tamanho da compra é a necessidade da empresa, e não a urgência da entrega.

In [29]:
# Exercício 5: Implementação de SCD Tipo 2
with engine.begin() as conn:
    # 1. Garantia da existencia das  colunas necessarias para SCD Tipo 2
    try:
        conn.execute(text("ALTER TABLE DIM_CLIENTE ADD COLUMN dt_inicio DATE DEFAULT '2011-01-01', ADD COLUMN dt_fim DATE DEFAULT NULL, ADD COLUMN flag_atual TINYINT(1) DEFAULT 1;"))
        conn.execute(text("UPDATE DIM_CLIENTE SET flag_atual = 1;"))
    except:
        pass

    # 2. SCD Tipo 2 atualiza o registro atual para Corporate
    conn.execute(text("""
        UPDATE DIM_CLIENTE 
        SET segmento = 'Corporate', 
            dt_inicio = CURRENT_DATE, 
            flag_atual = 1 
        WHERE nk_cliente IN ('AA-10315', 'AA-10375', 'AA-10480') 
        AND flag_atual = 1;
    """))

verificacao = """
SELECT nk_cliente, nome_cliente, segmento, dt_inicio, dt_fim, flag_atual 
FROM DIM_CLIENTE 
WHERE nk_cliente IN ('AA-10315', 'AA-10375', 'AA-10480')
ORDER BY nk_cliente, flag_atual DESC;
"""
display(pd.read_sql(verificacao, engine))

,nk_cliente,nome_cliente,segmento,dt_inicio,dt_fim,flag_atual
0,AA-10315,Alex Avila,Corporate,2026-06-28,None,1
1,AA-10375,Allen Armold,Corporate,2026-06-28,None,1
2,AA-10480,Andrew Allen,Corporate,2026-06-28,None,1


In [ ]:
# Exercício 6: Criação da View Executiva (2013)
# 1. Cria a View apenas uma vez (se ela não existir ou para garantir a estrutura correta)
with engine.begin() as conn:
    conn.execute(text("DROP VIEW IF EXISTS VW_PAINEL_EXECUTIVO;"))
    conn.execute(text("""
        CREATE VIEW VW_PAINEL_EXECUTIVO AS
        SELECT 
            t.ano,
            c.regiao,
            SUM(f.vlr_vendas) AS receita_total,
            SUM(f.vlr_lucro) AS lucro_total,
            (SUM(f.vlr_lucro) / NULLIF(SUM(f.vlr_vendas), 0)) * 100 AS margem_pct,
            SUM(f.vlr_vendas) / COUNT(DISTINCT f.nk_pedido) AS ticket_medio,
            (COUNT(DISTINCT CASE WHEN f.flag_retorno = 1 THEN f.nk_pedido END) / COUNT(DISTINCT f.nk_pedido)) * 100 AS taxa_devolucao_pct,
            COUNT(DISTINCT f.sk_cliente) AS qtd_clientes_unicos
        FROM FATO_VENDAS f
        JOIN DIM_TEMPO t ON f.sk_tempo = t.sk_tempo
        JOIN DIM_CLIENTE c ON f.sk_cliente = c.sk_cliente
        GROUP BY t.ano, c.regiao;
    """))

# 2. Consulta para 2013 irá mostrar um DataFrame vazio, pois não há dados para 2013
query_2013 = "SELECT regiao, ROUND(margem_pct, 2) as margem FROM VW_PAINEL_EXECUTIVO WHERE ano = 2013 ORDER BY margem_pct DESC;"
print("--- Resultado para 2013 (esperado vazio): ---")
display(pd.read_sql(query_2013, engine))

# 3. Consulta para 2014 que de fato possui dados
query_2014 = "SELECT regiao, ROUND(margem_pct, 2) as margem FROM VW_PAINEL_EXECUTIVO WHERE ano = 2014 ORDER BY margem_pct DESC;"
print("\n--- Resultado para 2014: ---")
display(pd.read_sql(query_2014, engine))

--- Resultado para 2013 (esperado vazio): ---


,regiao,margem



--- Resultado para 2014: ---


,regiao,margem
0,West,14.77
1,East,12.13
2,Central,7.47
3,South,4.06
